In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.denoising import *
from sklearn.preprocessing import StandardScaler
from scipy.linalg import svd
from scipy.spatial.distance import pdist, squareform, jaccard
from scipy.linalg import norm

In [ ]:
import scanpy as sc
import scvelo as scv

bdata = sc.read_h5ad("./data/pancreas/pancreas_inferred_velocity.h5ad")
scv.pl.velocity_embedding_stream(bdata, basis="umap", color="clusters", density=1.5, arrow_size=0.1)

In [ ]:
scv.pl.velocity_embedding_grid(bdata, basis="umap", color="clusters")

In [ ]:
from scripts.plotting import *
from scripts.denoising import *
from scripts.TPS import *

X_2d = bdata.obsm["X_umap"]
X = bdata.layers["Ms"]
tps = ThinPlateSpline(X_2d, n_control_points=2000)
tps.fit(X, dof_target=20)

In [ ]:
jacobians = tps.compute_tps_jacobians(X_2d)
jacobians.shape

In [ ]:
np.random.seed(42)
# Get number of genes
num_genes = X.shape[1]

# Randomly assign genes into 5 groups
n_clusters = 5
clusters = np.random.permutation(num_genes) % n_clusters

# Create a DataFrame to store cluster assignments
gene_clusters = pd.DataFrame({'Gene': np.arange(num_genes), 'Cluster': clusters})

In [ ]:
def project_velocities(Y, jacobians):
    """
    Projects the velocities using least squares regression on the Jacobians.

    Parameters:
    - Y: np.ndarray of shape (num_cells, num_genes), the observed velocity matrix.
    - jacobians: list of np.ndarray, each of shape (num_genes, D), the Jacobian matrices.

    Returns:
    - projected_velocities: np.ndarray of shape (num_cells, D), the estimated velocity projections.
    - intercepts: np.ndarray of shape (num_cells,), the intercepts from the least squares fits.
    - residuals: np.ndarray of shape (num_cells, num_genes), the residuals after projection.
    """
    num_cells, num_genes = Y.shape
    D = jacobians[0].shape[1]  # Number of features in each Jacobian

    # Initialize output arrays
    projected_velocities = np.zeros((num_cells, D))
#     betas = np.zeros(num_cells)
    residuals = np.zeros_like(Y)

    for i in range(num_cells):
        # Augment the Jacobian with a column of ones to capture the intercept
        J = jacobians[i]  # Shape: (num_genes, D)
#         J_aug = np.hstack([J, np.ones((J.shape[0], 1))])  # Shape: (num_genes, D+1)

        # Solve for parameters using least squares
        beta, _, _, _ = np.linalg.lstsq(J, Y[i], rcond=None)

        # Separate out the slope coefficients and intercept
#         beta = params[:-1]  # Shape: (D,)
        projected_velocities[i] = beta
#         intercepts[i] = intercept

        # Compute residuals
        predicted = J @ beta
        residuals[i] = Y[i] - predicted

    return projected_velocities, residuals

In [ ]:
def compute_r2_cells(Y, residuals, plot=False):
    """
    Compute R^2 scores for each cell.

    Parameters:
    - Y: np.ndarray of shape (num_cells, num_genes), observed velocity matrix.
    - residuals: np.ndarray of shape (num_cells, num_genes), residual matrix after projection.
    - plot: bool, whether to display a histogram of R^2 scores.

    Returns:
    - R2_scores_cell: np.ndarray of shape (num_cells,), R^2 values for each cell.
    """
    Y_mean_cell = np.mean(Y, axis=1)[:, np.newaxis]  # Mean velocity per cell
    SS_total_cell = np.sum((Y - Y_mean_cell) ** 2, axis=1)  # Total variance per cell
    SS_residual_cell = np.sum(residuals ** 2, axis=1)  # Sum of squared residuals per cell

    R2_scores_cell = 1 - (SS_residual_cell / SS_total_cell)  # Compute R^2 for each cell

    if plot:
        plt.figure(figsize=(8, 6))
        plt.hist(R2_scores_cell, bins=50, edgecolor='black', alpha=0.75)
        plt.xlabel("R^2 Score")
        plt.ylabel("Number of Cells")
        plt.title("Distribution of R^2 Scores Across Cells")
        plt.show()

    return R2_scores_cell


def compute_r2_genes(Y, residuals, plot=False):
    """
    Compute R^2 scores for each gene.

    Parameters:
    - Y: np.ndarray of shape (num_cells, num_genes), observed velocity matrix.
    - residuals: np.ndarray of shape (num_cells, num_genes), residual matrix after projection.
    - plot: bool, whether to display a histogram of R^2 scores.

    Returns:
    - R2_scores_gene: np.ndarray of shape (num_genes,), R^2 values for each gene.
    - R2_scores_gene_filtered: np.ndarray, R^2 values with absolute values <= 1.
    """
    Y_mean = np.mean(Y, axis=0)  # Mean velocity for each gene
    SS_total = np.sum((Y - Y_mean) ** 2, axis=0)  # Total variance
    SS_residual = np.sum(residuals ** 2, axis=0)  # Sum of squared residuals

    R2_scores_gene = 1 - (SS_residual / SS_total)  # Compute R^2 for each gene
    R2_scores_gene_filtered = R2_scores_gene[abs(R2_scores_gene) <= 1]  # Filter valid R^2 scores

    if plot:
        plt.figure(figsize=(8, 6))
        plt.hist(R2_scores_gene_filtered, bins=50, edgecolor='black', alpha=0.75)
        plt.xlabel("R^2 Score")
        plt.ylabel("Number of Genes")
        plt.title("Distribution of R^2 Scores Across Genes")
        plt.show()

    return R2_scores_gene, R2_scores_gene_filtered



def compute_top_r2_genes(Y, jacobians, beta, top_percent=10):
    """
    Computes residuals for all genes, calculates R^2 scores, and returns the top genes.

    Parameters:
    - Y: np.ndarray of shape (num_cells, num_genes), observed velocity matrix.
    - jacobians: list of np.ndarray, each of shape (num_genes, D), Jacobian matrices per cell.
    - beta: np.ndarray of shape (num_cells, D), regression coefficients per cell.
    - top_percent: int, percentage of top genes to return (default is 10%).

    Returns:
    - top_genes: np.ndarray, indices of the top genes based on R^2 scores.
    - R2_scores_gene: np.ndarray of shape (num_genes,), R^2 values for each gene.
    """
    num_cells, num_genes = Y.shape
    residuals = np.zeros_like(Y)  # Initialize residual matrix

    # Compute residuals for each gene
    for i in range(num_cells):
        J = jacobians[i]  # Jacobian for cell i
        predicted = J @ beta[i]  # Predicted velocity
        residuals[i] = Y[i] - predicted  # Compute residual

    # Compute R² scores for each gene
    Y_mean = np.mean(Y, axis=0)  # Mean velocity for each gene
    SS_total = np.sum((Y - Y_mean) ** 2, axis=0)  # Total variance
    SS_residual = np.sum(residuals ** 2, axis=0)  # Sum of squared residuals
    R2_scores_gene = 1 - (SS_residual / SS_total)  # Compute R² for each gene

    # Get the top X% genes based on R² scores
    num_top_genes = int(len(R2_scores_gene) * (top_percent / 100))
    top_genes = np.argsort(R2_scores_gene)[-num_top_genes:]  # Indices of top genes

    return top_genes, R2_scores_gene

In [ ]:
gene_names = np.array(bdata.var_names)

In [ ]:
Y = bdata.layers["velocity"]
first_gene_group_indices = (clusters == 0)

# Step 1: Select only the first gene group
Y_subset = Y[:, first_gene_group_indices]  # Restrict to selected genes
jacobians_subset = jacobians[:, first_gene_group_indices, :]  # Restrict Jacobians

for iteration in range(5):
    print(f"Iteration {iteration + 1}: Y_subset shape {Y_subset.shape}")

    # Step 2: Apply projection
    projected_velocities, residuals_subset = project_velocities(Y_subset, jacobians_subset)

    # Step 3: Compute top genes based on R² scores
    top_genes, R2_scores_gene = compute_top_r2_genes(Y, jacobians, 
                                                     projected_velocities, top_percent=10)

    # Step 4: Subset Y and Jacobians for the next iteration
    Y_subset = Y[:, top_genes]  # Restrict Y to top genes
    jacobians_subset = jacobians[:, top_genes, :]  # Restrict Jacobians

    print(f"Selected {len(top_genes)} genes for the next iteration")

# Final selected gene indices after 5 iterations
final_gene_names = gene_names[top_genes]
print(final_gene_names)


R2_scores_cell = compute_r2_cells(Y_subset, residuals_subset, plot=True)
R2_scores_gene, R2_scores_gene_filtered = compute_r2_genes(Y_subset, residuals_subset, plot=True)

In [ ]:
cluster_labels = bdata.obs['clusters'].astype(str).values  # Ensure string type for mapping

# Extract cluster colors
cluster_colors = bdata.uns['clusters_colors']  # List of colors indexed by cluster order

# Get unique cluster names in the order stored in 'clusters_colors'
unique_clusters = np.unique(cluster_labels)

# Create a mapping from cluster names to colors
cluster_to_color = {cluster: color for cluster, color in zip(unique_clusters, cluster_colors)}

# Assign colors to each cell based on its cluster
cell_colors = [cluster_to_color[cluster] for cluster in cluster_labels]

In [ ]:
plot_2d_quiver(X_2d, projected_velocities, cell_colors, scale=5, cmap='coolwarm', 
               arrow_color='black', use_normalized=True)

In [ ]:
tps_vf = ThinPlateSpline(X_2d, n_control_points=2000)
tps_vf.fit(projected_velocities, dof_target=50)

In [ ]:
vf_2d_smoothed = tps_vf.predict(X_2d)
plot_2d_quiver(X_2d, vf_2d_smoothed, cell_colors, scale=4, cmap='coolwarm', arrow_color='black', use_normalized=True)

In [ ]:
from scripts.steam_plot import *


plot_velocity_streamplot(X_2d, tps_vf, cell_colors, 20)

In [ ]:
first_gene_group_indices = (clusters == 1)

# Step 1: Select only the first gene group
Y_subset = Y[:, first_gene_group_indices]  # Restrict to selected genes
jacobians_subset = jacobians[:, first_gene_group_indices, :]  # Restrict Jacobians

for iteration in range(5):
    print(f"Iteration {iteration + 1}: Y_subset shape {Y_subset.shape}")

    # Step 2: Apply projection
    projected_velocities, residuals_subset = project_velocities(Y_subset, jacobians_subset)

    # Step 3: Compute top genes based on R² scores
    top_genes, R2_scores_gene = compute_top_r2_genes(Y, jacobians, 
                                                     projected_velocities, top_percent=10)

    # Step 4: Subset Y and Jacobians for the next iteration
    Y_subset = Y[:, top_genes]  # Restrict Y to top genes
    jacobians_subset = jacobians[:, top_genes, :]  # Restrict Jacobians

    print(f"Selected {len(top_genes)} genes for the next iteration")

# Final selected gene indices after 5 iterations
final_gene_names = gene_names[top_genes]
print(final_gene_names)

R2_scores_cell = compute_r2_cells(Y_subset, residuals_subset, plot=True)
R2_scores_gene, R2_scores_gene_filtered = compute_r2_genes(Y_subset, residuals_subset, plot=True)

tps_vf = ThinPlateSpline(X_2d, n_control_points=2000)
tps_vf.fit(projected_velocities, dof_target=50)
plot_velocity_streamplot(X_2d, tps_vf, cell_colors, 20)

In [ ]:
first_gene_group_indices = (clusters == 2)

# Step 1: Select only the first gene group
Y_subset = Y[:, first_gene_group_indices]  # Restrict to selected genes
jacobians_subset = jacobians[:, first_gene_group_indices, :]  # Restrict Jacobians

for iteration in range(5):
    print(f"Iteration {iteration + 1}: Y_subset shape {Y_subset.shape}")

    # Step 2: Apply projection
    projected_velocities, residuals_subset = project_velocities(Y_subset, jacobians_subset)

    # Step 3: Compute top genes based on R² scores
    top_genes, R2_scores_gene = compute_top_r2_genes(Y, jacobians, 
                                                     projected_velocities, top_percent=10)

    # Step 4: Subset Y and Jacobians for the next iteration
    Y_subset = Y[:, top_genes]  # Restrict Y to top genes
    jacobians_subset = jacobians[:, top_genes, :]  # Restrict Jacobians

    print(f"Selected {len(top_genes)} genes for the next iteration")

# Final selected gene indices after 5 iterations
final_gene_names = gene_names[top_genes]
print(final_gene_names)

R2_scores_cell = compute_r2_cells(Y_subset, residuals_subset, plot=True)
R2_scores_gene, R2_scores_gene_filtered = compute_r2_genes(Y_subset, residuals_subset, plot=True)

tps_vf = ThinPlateSpline(X_2d, n_control_points=2000)
tps_vf.fit(projected_velocities, dof_target=50)
plot_velocity_streamplot(X_2d, tps_vf, cell_colors, 20)

In [ ]:
first_gene_group_indices = (clusters == 3)

# Step 1: Select only the first gene group
Y_subset = Y[:, first_gene_group_indices]  # Restrict to selected genes
jacobians_subset = jacobians[:, first_gene_group_indices, :]  # Restrict Jacobians

for iteration in range(5):
    print(f"Iteration {iteration + 1}: Y_subset shape {Y_subset.shape}")

    # Step 2: Apply projection
    projected_velocities, residuals_subset = project_velocities(Y_subset, jacobians_subset)

    # Step 3: Compute top genes based on R² scores
    top_genes, R2_scores_gene = compute_top_r2_genes(Y, jacobians, 
                                                     projected_velocities, top_percent=10)

    # Step 4: Subset Y and Jacobians for the next iteration
    Y_subset = Y[:, top_genes]  # Restrict Y to top genes
    jacobians_subset = jacobians[:, top_genes, :]  # Restrict Jacobians

    print(f"Selected {len(top_genes)} genes for the next iteration")

# Final selected gene indices after 5 iterations
final_gene_names = gene_names[top_genes]
print(final_gene_names)

R2_scores_cell = compute_r2_cells(Y_subset, residuals_subset, plot=True)
R2_scores_gene, R2_scores_gene_filtered = compute_r2_genes(Y_subset, residuals_subset, plot=True)

tps_vf = ThinPlateSpline(X_2d, n_control_points=2000)
tps_vf.fit(projected_velocities, dof_target=50)
plot_velocity_streamplot(X_2d, tps_vf, cell_colors, 20)

In [ ]:
first_gene_group_indices = (clusters == 4)

# Step 1: Select only the first gene group
Y_subset = Y[:, first_gene_group_indices]  # Restrict to selected genes
jacobians_subset = jacobians[:, first_gene_group_indices, :]  # Restrict Jacobians

for iteration in range(5):
    print(f"Iteration {iteration + 1}: Y_subset shape {Y_subset.shape}")

    # Step 2: Apply projection
    projected_velocities, residuals_subset = project_velocities(Y_subset, jacobians_subset)

    # Step 3: Compute top genes based on R² scores
    top_genes, R2_scores_gene = compute_top_r2_genes(Y, jacobians, 
                                                     projected_velocities, top_percent=10)

    # Step 4: Subset Y and Jacobians for the next iteration
    Y_subset = Y[:, top_genes]  # Restrict Y to top genes
    jacobians_subset = jacobians[:, top_genes, :]  # Restrict Jacobians

    print(f"Selected {len(top_genes)} genes for the next iteration")

# Final selected gene indices after 5 iterations
final_gene_names = gene_names[top_genes]
print(final_gene_names)

R2_scores_cell = compute_r2_cells(Y_subset, residuals_subset, plot=True)
R2_scores_gene, R2_scores_gene_filtered = compute_r2_genes(Y_subset, residuals_subset, plot=True)

tps_vf = ThinPlateSpline(X_2d, n_control_points=2000)
tps_vf.fit(projected_velocities, dof_target=50)
plot_velocity_streamplot(X_2d, tps_vf, cell_colors, 20)

In [ ]:
first_gene_group_indices = (clusters == 5)

# Step 1: Select only the first gene group
Y_subset = Y[:, first_gene_group_indices]  # Restrict to selected genes
jacobians_subset = jacobians[:, first_gene_group_indices, :]  # Restrict Jacobians

for iteration in range(5):
    print(f"Iteration {iteration + 1}: Y_subset shape {Y_subset.shape}")

    # Step 2: Apply projection
    projected_velocities, residuals_subset = project_velocities(Y_subset, jacobians_subset)

    # Step 3: Compute top genes based on R² scores
    top_genes, R2_scores_gene = compute_top_r2_genes(Y, jacobians, 
                                                     projected_velocities, top_percent=10)

    # Step 4: Subset Y and Jacobians for the next iteration
    Y_subset = Y[:, top_genes]  # Restrict Y to top genes
    jacobians_subset = jacobians[:, top_genes, :]  # Restrict Jacobians

    print(f"Selected {len(top_genes)} genes for the next iteration")

# Final selected gene indices after 5 iterations
final_gene_names = gene_names[top_genes]
print(final_gene_names)

R2_scores_cell = compute_r2_cells(Y_subset, residuals_subset, plot=True)
R2_scores_gene, R2_scores_gene_filtered = compute_r2_genes(Y_subset, residuals_subset, plot=True)

tps_vf = ThinPlateSpline(X_2d, n_control_points=2000)
tps_vf.fit(projected_velocities, dof_target=50)
plot_velocity_streamplot(X_2d, tps_vf, cell_colors, 20)